# Codice per generare una semplice simuazione Geant4 tramite un LLM

In [ ]:
# =====================================================
# Cell 0 — API import and connection test
# =====================================================

# Import
from google import genai

# API key
API_KEY = "Insert_here_your_API_key"

# Client Gemini
client = genai.Client(api_key=API_KEY)

# Test della connessione
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Reply with exactly: API connection successful"
)

# Stampa della risposta
print(response.text)

API connection successful


In [24]:
# ============================================================
# Cell 1 — Generate IR from user prompt
# ============================================================

import json
from pathlib import Path

from backend.knowledge_base import (
    load_knowledge_base,
    EXAMPLES_DIR,
)

from backend.prompt_builder import build_prompt


# ============================================================
# LOAD KNOWLEDGE BASE
# ============================================================

knowledge_base = load_knowledge_base(EXAMPLES_DIR)


# ============================================================
# USER PROMPT
# ============================================================

user_prompt = input("Geant4 request: ")


# ============================================================
# BUILD COMPLETE PROMPT
# ============================================================

prompt = build_prompt(
    knowledge_base,
    user_prompt
)


# ============================================================
# GENERATE JSON IR
# ============================================================

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt,
    config={
        "response_mime_type": "application/json"
    }
)


# ============================================================
# PARSE JSON
# ============================================================

ir = json.loads(response.text)


# ============================================================
# CREATE OUTPUT DIRECTORY
# ============================================================

project_root = EXAMPLES_DIR.parent

output_dir = project_root / "json_from_Gemini"
output_dir.mkdir(exist_ok=True)


# ============================================================
# SAVE JSON
# ============================================================

output_file = output_dir / "generated_ir.json"

with output_file.open("w", encoding="utf-8") as f:
    json.dump(
        ir,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# OUTPUT
# ============================================================

print("\n" + "=" * 60)
print("GENERATED IR")
print("=" * 60)

print(json.dumps(
    ir,
    indent=2,
    ensure_ascii=False
))

print("\n" + "=" * 60)
print("SAVED TO")
print("=" * 60)

print(output_file)


GENERATED IR
{
  "geometry": {
    "primitives": {
      "sphere": [
        {
          "name": "WaterSphere",
          "radius": {
            "value": 5.0,
            "unit": "cm"
          },
          "material": "G4_WATER"
        }
      ],
      "box": [
        {
          "name": "SiliconBox",
          "half_size": [
            {
              "value": 1.0,
              "unit": "cm"
            },
            {
              "value": 1.0,
              "unit": "cm"
            },
            {
              "value": 1.0,
              "unit": "cm"
            }
          ],
          "material": "G4_Si"
        }
      ]
    },
    "placement": [
      {
        "type": "place",
        "child": "WaterSphere",
        "mother": "worldLV",
        "position": [
          {
            "value": 0.0,
            "unit": "cm"
          },
          {
            "value": 0.0,
            "unit": "cm"
          },
          {
            "value": 0.0,
            "unit": "cm

In [25]:
# =====================================================
# Cell 1 — Environment & imports
# =====================================================

import sys
from pathlib import Path

# --- Make project root visible ---
PROJECT_ROOT = Path().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# --- IR pipeline ---
from backend.ir.validator import validate_ir
from backend.ir.units import length, energy, vector3
from backend.compiler.compile_ir import compile_ir

# --- Run control ---
from backend.run_control import (
    set_sim_name,
    set_num_events,
)

# --- Build & run ---
from backend.runner import (
    build,
    run_simulation,
)

print("✔ Environment and imports OK")

✔ Environment and imports OK


In [26]:
# =====================================================
# Cell 2 — Load IR from JSON
# =====================================================

import json

IR_PATH = "json_from_Gemini/generated_ir.json"   # cambia path
# IR_PATH = "validation_set/CylindricalHole.json" 

with open(IR_PATH, "r") as f:
    ir = json.load(f)

validate_ir(ir)
print("✔ IR validation passed")

compile_ir(ir)
print("✔ IR compiled to C++")

✔ IR validation passed
✔ IR compiled to C++


In [27]:
# =====================================================
# Cell 3 — Set parameters
# =====================================================

from backend.run_control import (
    set_sim_name,
    set_num_events,
)

from backend.runner import build, run_simulation
from backend.compiler.compile_ir import compile_ir

# --- Simulation parameters ---
set_sim_name("prova")
set_num_events(100000)

# --- Compile IR → C++ ---
compile_ir(ir)

print("✔ IR compiled into Geant4 sources")


✔ IR compiled into Geant4 sources


In [28]:
# =====================================================
# Cell 4 — Build
# =====================================================

build()

Consolidate compiler generated dependencies of target SimpleSim
[ 12%] Building CXX object CMakeFiles/SimpleSim.dir/src/DetectorConstruction.cc.o
[ 25%] Building CXX object CMakeFiles/SimpleSim.dir/src/ActionInitialization.cc.o
[ 37%] Linking CXX executable SimpleSim
[100%] Built target SimpleSim


In [29]:
# =====================================================
# Cell 5 — run (with visualization)
# =====================================================

run_simulation(vis=True)


**************************************************************
 Geant4 version Name: geant4-11-01-patch-02 [MT]   (15-June-2023)
                       Copyright : Geant4 Collaboration
                      References : NIM A 506 (2003), 250-303
                                 : IEEE-TNS 53 (2006), 270-278
                                 : NIM A 835 (2016), 186-225
                             WWW : http://geant4.org/
**************************************************************

<<< Geant4 Physics List simulation engine: FTFP_BERT_HP

G4VModularPhysicsList::ReplacePhysics: G4EmStandard with type : 2 is replaced with G4EmStandard_opt4
Checking overlaps for volume WaterSpherePV:0 (G4Sphere) ... OK! 
Checking overlaps for volume SiliconBoxPV:0 (G4Box) ... OK! 

 hInelastic FTFP_BERT_HP : threshold between BERT and FTFP is over the interval 
 for pions :   3 to 6 GeV
 for kaons :   3 to 6 GeV
 for proton :  3 to 6 GeV
 for neutron : 3 to 6 GeV

/home/emanuelemazzola/geant4-11.1.2/inst

In [ ]:
# =====================================================
# Cell 6 — run (without visualization)
# =====================================================

run_simulation(vis=False)

In [ ]:
# =====================================================
# Cell — Validate ALL partial examples (generic)
# =====================================================

import json
from pathlib import Path

from backend.ir.validator import (
    validate_geometry,
    validate_source,
)

# -----------------------------------------------------
#  SELECT WHAT TO VALIDATE
# -----------------------------------------------------
SECTION = "source"   # "geometry" | "source" | "scoring"

BASE_DIR = Path(f"examples/{SECTION}")

# -----------------------------------------------------
#  Validator dispatch
# -----------------------------------------------------
def validate_section(data, section):
    if section not in data:
        raise ValueError(f"Missing '{section}' section")

    if section == "geometry":
        validate_geometry(data["geometry"])

    elif section == "source":
        validate_source(data["source"])

    elif section == "scoring":
        # scoring è semplice: solo lista
        if not isinstance(data["scoring"], list):
            raise ValueError("Scoring must be a list")

    else:
        raise ValueError(f"Unknown section '{section}'")


# -----------------------------------------------------
#  File validator
# -----------------------------------------------------
def validate_partial_file(path, section):
    try:
        with open(path, "r") as f:
            data = json.load(f)

        validate_section(data, section)

        print(f"✔ OK: {path.relative_to(BASE_DIR)}")
        return True

    except Exception as e:
        print(f"✘ FAIL: {path.relative_to(BASE_DIR)}")
        print(f"    {type(e).__name__}: {e}")
        return False


# -----------------------------------------------------
#  Runner
# -----------------------------------------------------
def run():
    files = sorted(BASE_DIR.rglob(f"{SECTION}.json"))

    print(f"Validating '{SECTION}' examples")
    print(f"Found {len(files)} files\n")

    ok = 0
    fail = 0

    for f in files:
        if validate_partial_file(f, SECTION):
            ok += 1
        else:
            fail += 1

    print("\n--- Summary ---")
    print(f"✔ Passed: {ok}")
    print(f"✘ Failed: {fail}")


run()